# RSA Significance Maps

Two sections:

1. **Group-average** — analytical p-values from Spearman ρ (t-approximation), uncorrected + FDR-corrected maps, borders around significant clusters (≥10 vertices)
2. **Per-subject random-effects** — Fisher-z → one-sample t-test per vertex, Cohen's d map, significance maps with borders

**Environment:** `analysis`  
**Input:** `rsa_spearman.dscalar.nii` from `rsa/run_analysis.sh` outputs

In [ ]:
from pathlib import Path

import nibabel as nib
import numpy as np
import matplotlib.pyplot as plt
from nilearn import plotting, surface
from scipy import stats
from scipy.sparse import csr_matrix
from scipy.sparse.csgraph import connected_components
from statsmodels.stats.multitest import fdrcorrection

In [ ]:
# ── CONFIG ────────────────────────────────────────────────────────────────────
DATA_BASE    = Path("/home/amin/Research/Representation/Movie/data/Setareh")
OUTPUTS_BASE = Path("/home/amin/Research/Representation/Movie/outputs")
HCP_DIR      = DATA_BASE / "HCP_S1200_GroupAvg_v1"

RSA_DIR = OUTPUTS_BASE / "rsa"

# CIFTI template — preprocessed group-average dtseries; its BrainModelAxis
# defines the 59k cortex structure (54216 LH + 54225 RH = 108441 grayordinates,
# vertex indices in [0, 59291] on the 59k surface).
TEMPLATE_CIFTI = str(OUTPUTS_BASE / "preprocessed" /
                     "group_average_gsr_zscore_cortex_59k.dtseries.nii")

# 59k midthickness — the reference surface for the CIFTI grayordinates
LEFT_SURF  = str(HCP_DIR / "S1200.L.midthickness_MSMAll.59k_fs_LR.surf.gii")
RIGHT_SURF = str(HCP_DIR / "S1200.R.midthickness_MSMAll.59k_fs_LR.surf.gii")

# Full vertex count for 59k surfaces (includes medial wall)
N_VERTS_LH = 59292
N_VERTS_RH = 59292

# Sphere surfaces for 59k→32k resampling (pycortex section)
SPHERE_59K_LH = str(HCP_DIR / "S1200.L.sphere.59k_fs_LR.surf.gii")
SPHERE_59K_RH = str(HCP_DIR / "S1200.R.sphere.59k_fs_LR.surf.gii")
SPHERE_32K_LH = str(HCP_DIR / "S1200.L.sphere.32k_fs_LR.surf.gii")
SPHERE_32K_RH = str(HCP_DIR / "S1200.R.sphere.32k_fs_LR.surf.gii")
TEMPLATE_32K  = str(HCP_DIR / "S1200.curvature_MSMAll.32k_fs_LR.dscalar.nii")

WORKBENCH = "/opt/workbench/bin_linux64/wb_command"

MODEL    = "pe-av-small-16-frame"
MODALITY = "av"
CONFIG   = "k100_norm_delay5s_bin2s_spearman"
METHOD   = "spearman"

K       = 100
N_PAIRS = K * (K - 1) // 2   # = 4950

ALPHA            = 0.05
MIN_CLUSTER_SIZE = 10
# ─────────────────────────────────────────────────────────────────────────────

GROUP_DIR = RSA_DIR / "group_average" / MODEL / CONFIG
print(f"Group RSA dir: {GROUP_DIR}")
print(f"N_PAIRS per searchlight: {N_PAIRS}")

## Helper Functions

In [ ]:
def load_cifti_map(path: str) -> np.ndarray:
    """Load a CIFTI dscalar → (n_grayords,) float32."""
    return nib.load(path).get_fdata(dtype=np.float32).squeeze()


def save_cifti_map(data: np.ndarray, template_path: str, out_path: str,
                   map_name: str = "map") -> None:
    """Save (n_grayords,) array as CIFTI dscalar.nii."""
    ref = nib.load(template_path)
    bm_axis  = ref.header.get_axis(1)
    sc_axis  = nib.cifti2.ScalarAxis([map_name])
    header   = nib.cifti2.Cifti2Header.from_axes((sc_axis, bm_axis))
    img = nib.Cifti2Image(data.astype(np.float32).reshape(1, -1), header=header)
    nib.save(img, out_path)
    print(f"Saved: {out_path}")


def cifti_to_surface_arrays(data: np.ndarray, template_path: str,
                             n_verts_lh: int, n_verts_rh: int):
    """Expand full-brain CIFTI data to per-hemisphere surface arrays.

    CIFTI grayordinates cover only non-medial-wall vertices.  This function
    creates full-surface arrays (including medial wall) and maps values to
    their correct vertex positions using the CIFTI BrainModelAxis.

    Args:
        data:          (n_grayords,) float32 — full-brain CIFTI values
        template_path: path to CIFTI file with the BrainModelAxis to use
        n_verts_lh:    total vertices in the left 59k surface GIFTI
        n_verts_rh:    total vertices in the right 59k surface GIFTI

    Returns:
        lh_full:   (n_verts_lh,) float32 — NaN at medial wall
        rh_full:   (n_verts_rh,) float32 — NaN at medial wall
        lh_vidx:   vertex indices of LH grayordinates in the 59k mesh
        rh_vidx:   vertex indices of RH grayordinates in the 59k mesh
    """
    bm = nib.load(template_path).header.get_axis(1)
    lh_full = np.full(n_verts_lh, np.nan, dtype=np.float32)
    rh_full = np.full(n_verts_rh, np.nan, dtype=np.float32)
    lh_vidx = rh_vidx = None

    for name, sl, struct_bm in bm.iter_structures():
        vidx = struct_bm.vertex
        if "CORTEX_LEFT"  in name:
            lh_full[vidx] = data[sl]
            lh_vidx = vidx
        elif "CORTEX_RIGHT" in name:
            rh_full[vidx] = data[sl]
            rh_vidx = vidx

    return lh_full, rh_full, lh_vidx, rh_vidx


def rho_to_pvalue(rho: np.ndarray, n_pairs: int,
                  alternative: str = "greater") -> np.ndarray:
    """Analytical p-value for Spearman ρ via t-approximation.

    t = ρ * sqrt((n-2) / (1 - ρ²))  ~  t(df = n-2)

    Args:
        rho:         (n_vertices,) correlation values (CIFTI grayords)
        n_pairs:     unique pairwise distances per searchlight = K*(K-1)/2
        alternative: 'greater' (one-tailed, positive ρ) or 'two-sided'
    """
    r = np.clip(rho, -1 + 1e-10, 1 - 1e-10).astype(np.float64)
    t = r * np.sqrt((n_pairs - 2) / (1.0 - r ** 2))
    if alternative == "greater":
        p = stats.t.sf(t, df=n_pairs - 2)
    else:
        p = 2.0 * stats.t.sf(np.abs(t), df=n_pairs - 2)
    return p.astype(np.float32)


def load_surface_faces(surf_gifti_path: str) -> np.ndarray:
    """Return (n_faces, 3) int32 face array from a GIFTI surface file."""
    surf = nib.load(surf_gifti_path)
    return surf.darrays[1].data.astype(np.int32)


def cifti_surface_adjacency(faces_surf: np.ndarray,
                             vertex_idx: np.ndarray,
                             n_cifti_verts: int) -> csr_matrix:
    """Sparse adjacency matrix among CIFTI grayordinate vertices.

    Re-indexes the full-surface face array to CIFTI vertex space, keeping
    only faces whose three vertices are all grayordinates.

    Args:
        faces_surf:    (n_faces, 3) faces from the 59k surface mesh
        vertex_idx:    (n_cifti_verts,) vertex indices of CIFTI grayordinates
        n_cifti_verts: number of CIFTI grayordinates for this hemisphere
    """
    # Map 59k vertex index → CIFTI index (-1 if not a grayordinate)
    idx_map = np.full(int(faces_surf.max()) + 1, -1, dtype=np.int32)
    idx_map[vertex_idx] = np.arange(n_cifti_verts, dtype=np.int32)

    f0 = idx_map[faces_surf[:, 0]]
    f1 = idx_map[faces_surf[:, 1]]
    f2 = idx_map[faces_surf[:, 2]]
    valid = (f0 >= 0) & (f1 >= 0) & (f2 >= 0)

    fc = np.column_stack([f0[valid], f1[valid], f2[valid]])
    i = np.concatenate([fc[:, 0], fc[:, 1], fc[:, 2],
                        fc[:, 1], fc[:, 2], fc[:, 0]])
    j = np.concatenate([fc[:, 1], fc[:, 2], fc[:, 0],
                        fc[:, 0], fc[:, 1], fc[:, 2]])
    return csr_matrix((np.ones(len(i), dtype=np.uint8), (i, j)),
                      shape=(n_cifti_verts, n_cifti_verts))


def find_cluster_borders(sig_mask: np.ndarray, adj: csr_matrix,
                         min_cluster_size: int = 10) -> np.ndarray:
    """Border vertices of significant clusters with ≥ min_cluster_size vertices.

    A significant vertex is a border if ≥1 immediate mesh neighbour is
    non-significant.  Only clusters meeting the size threshold are included.

    Args:
        sig_mask: (n_cifti_verts,) bool
        adj:      sparse adjacency in CIFTI vertex space
        min_cluster_size: minimum number of vertices per cluster
    """
    if not sig_mask.any():
        return np.zeros(len(sig_mask), dtype=bool)

    sig_idx = np.where(sig_mask)[0]
    sig_adj = adj[sig_idx][:, sig_idx]
    n_comp, labels = connected_components(sig_adj, directed=False)

    border = np.zeros(len(sig_mask), dtype=bool)
    for comp in range(n_comp):
        comp_local  = np.where(labels == comp)[0]
        comp_global = sig_idx[comp_local]
        if len(comp_global) < min_cluster_size:
            continue
        for v in comp_global:
            nbrs = adj[v].indices
            if np.any(~sig_mask[nbrs]):
                border[v] = True

    return border


print("Helpers defined.")

In [ ]:
def plot_surface_bilateral(lh_data: np.ndarray, rh_data: np.ndarray,
                            title: str, threshold: float = None,
                            cmap: str = "cold_hot", vmax: float = None,
                            bg_on_stat: bool = True) -> None:
    """Lateral + medial views, both hemispheres.

    lh_data / rh_data: full-surface arrays (59292,), NaN at medial wall.
    Uses the 59k midthickness as display surface (inflated not available at 59k).
    """
    fig, axes = plt.subplots(2, 4, subplot_kw={"projection": "3d"}, figsize=(18, 8))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    hems  = [(lh_data, LEFT_SURF,  "left",  "Left"),
             (rh_data, RIGHT_SURF, "right", "Right")]
    views = ["lateral", "medial"]

    for row, view in enumerate(views):
        for col, (data, surf, hemi_str, label) in enumerate(hems):
            ax = axes[row, col * 2 + (0 if view == "lateral" else 1)]
            # Use 99th-percentile of non-NaN values for colour scale
            vals = np.abs(data[~np.isnan(data)])
            vmax_ = vmax or (float(np.percentile(vals, 99)) if len(vals) else 1.0)
            th    = threshold if threshold is not None else 0.0

            plotting.plot_surf_stat_map(
                surf_mesh=surf, stat_map=data,
                threshold=th, cmap=cmap, vmax=vmax_,
                view=view, hemi=hemi_str,
                colorbar=(row == 0 and col == 1),
                bg_on_stat=bg_on_stat, axes=ax,
            )
            ax.set_title(f"{label} {view}", fontsize=9)

    plt.tight_layout()
    plt.show()
    plt.close(fig)


def _expand_border_to_surface(border_mask: np.ndarray, vertex_idx: np.ndarray,
                               n_surf_verts: int) -> np.ndarray:
    """Map per-CIFTI-vertex border mask back to full surface space."""
    out = np.zeros(n_surf_verts, dtype=np.float32)
    out[vertex_idx] = border_mask.astype(np.float32)
    return out


def plot_surface_with_borders(lh_data: np.ndarray, rh_data: np.ndarray,
                               lh_border_cifti: np.ndarray,
                               rh_border_cifti: np.ndarray,
                               lh_vidx: np.ndarray, rh_vidx: np.ndarray,
                               title: str, threshold: float = None,
                               cmap: str = "cold_hot", vmax: float = None) -> None:
    """Bilateral map with black borders around significant clusters."""
    n_lh = lh_data.shape[0]
    n_rh = rh_data.shape[0]
    lh_border_surf = _expand_border_to_surface(lh_border_cifti, lh_vidx, n_lh)
    rh_border_surf = _expand_border_to_surface(rh_border_cifti, rh_vidx, n_rh)

    fig, axes = plt.subplots(2, 4, subplot_kw={"projection": "3d"}, figsize=(18, 8))
    fig.suptitle(title, fontsize=13, fontweight="bold")

    hems  = [(lh_data, lh_border_surf, LEFT_SURF,  "left",  "Left"),
             (rh_data, rh_border_surf, RIGHT_SURF, "right", "Right")]
    views = ["lateral", "medial"]

    for row, view in enumerate(views):
        for col, (data, border_s, surf, hemi_str, label) in enumerate(hems):
            ax = axes[row, col * 2 + (0 if view == "lateral" else 1)]
            vals  = np.abs(data[~np.isnan(data)])
            vmax_ = vmax or (float(np.percentile(vals, 99)) if len(vals) else 1.0)
            th    = threshold if threshold is not None else 0.0

            plotting.plot_surf_stat_map(
                surf_mesh=surf, stat_map=data,
                threshold=th, cmap=cmap, vmax=vmax_,
                view=view, hemi=hemi_str,
                colorbar=False, bg_on_stat=True, axes=ax,
            )
            # Border: show only border vertices in black
            if border_s.any():
                plotting.plot_surf_stat_map(
                    surf_mesh=surf, stat_map=border_s,
                    threshold=0.5, cmap="Greys", vmax=1.0,
                    view=view, hemi=hemi_str,
                    colorbar=False, bg_on_stat=False, alpha=1.0, axes=ax,
                )
            ax.set_title(f"{label} {view}", fontsize=9)

    plt.tight_layout()
    plt.show()
    plt.close(fig)


print("Plot helpers defined.")

## Group-Average RSA Significance Maps

Analytical p-values via t-approximation: `t = ρ√((n-2)/(1-ρ²))`, df = n-2  
where n = K×(K-1)/2 = 4950 unique pairwise distances per searchlight.

In [ ]:
# Load group-average RSA correlation map
rho_path = GROUP_DIR / f"rsa_{METHOD}.dscalar.nii"
rho_full  = load_cifti_map(str(rho_path))
print(f"ρ map: {rho_full.shape}  max={rho_full.max():.4f}  mean={rho_full.mean():.4f}")

# Expand to full-surface arrays; record grayordinate vertex indices per hemisphere
lh_rho, rh_rho, lh_vidx, rh_vidx = cifti_to_surface_arrays(
    rho_full, TEMPLATE_CIFTI, N_VERTS_LH, N_VERTS_RH
)
n_lh = len(lh_vidx)   # 54216 (non-medial-wall LH grayordinates)
n_rh = len(rh_vidx)   # 54225 (non-medial-wall RH grayordinates)
print(f"LH surface: {lh_rho.shape}  RH surface: {rh_rho.shape}")
print(f"LH grayordinates: {n_lh}  RH grayordinates: {n_rh}")

In [ ]:
# ── Analytical p-values (one-tailed: ρ > 0) ──────────────────────────────────
pval_uncorr = rho_to_pvalue(rho_full, N_PAIRS, alternative="greater")
print(f"Uncorrected: n(p<0.05) = {(pval_uncorr < ALPHA).sum():,} / {len(pval_uncorr):,}")

# Whole-brain FDR correction over all 108441 grayordinates
_, pval_fdr = fdrcorrection(pval_uncorr)
pval_fdr = pval_fdr.astype(np.float32)
print(f"FDR q<0.05:  n = {(pval_fdr < ALPHA).sum():,}")

In [ ]:
# ── Save group-average significance maps as CIFTI dscalar (for wb_view) ──────
# Load in wb_view with the 59k midthickness surface:
#   File → Open File → select .dscalar.nii
#   File → Open File → select S1200.L/R.midthickness_MSMAll.59k_fs_LR.surf.gii
# For inflated views, generate 59k inflated surfaces once with wb_command:
#   wb_command -surface-generate-inflated \
#       S1200.L.midthickness_MSMAll.59k_fs_LR.surf.gii \
#       S1200.L.inflated_MSMAll.59k_fs_LR.surf.gii \
#       S1200.L.very_inflated_MSMAll.59k_fs_LR.surf.gii \
#       -iterations-scale 2.5

save_cifti_map(pval_uncorr, TEMPLATE_CIFTI,
               str(GROUP_DIR / "pval_uncorr.dscalar.nii"), "pval_uncorr")
save_cifti_map(pval_fdr, TEMPLATE_CIFTI,
               str(GROUP_DIR / "pval_fdr.dscalar.nii"),    "pval_fdr")

# Masked rho maps (zeros outside threshold — convenient for wb_view colour scale)
rho_uncorr_sig = (rho_full * (pval_uncorr < ALPHA)).astype(np.float32)
rho_fdr_sig    = (rho_full * (pval_fdr    < ALPHA)).astype(np.float32)
save_cifti_map(rho_uncorr_sig, TEMPLATE_CIFTI,
               str(GROUP_DIR / f"rsa_{METHOD}_uncorr_sig.dscalar.nii"), "rsa_uncorr_sig")
save_cifti_map(rho_fdr_sig,    TEMPLATE_CIFTI,
               str(GROUP_DIR / f"rsa_{METHOD}_fdr_sig.dscalar.nii"),    "rsa_fdr_sig")

In [ ]:
# ── Plot: ρ map (no threshold) ────────────────────────────────────────────────
plot_surface_bilateral(
    lh_rho, rh_rho,
    title=f"{MODEL} / {MODALITY} — Spearman ρ (group average)",
    threshold=0.0, cmap="cold_hot",
)

In [ ]:
# ── Plot: uncorrected significance map ───────────────────────────────────────
sig_mask_uncorr = pval_uncorr < ALPHA  # (59412,) CIFTI space

# Full-surface arrays: show ρ where significant, 0 elsewhere
lh_sig_uncorr = np.full(N_VERTS_LH, 0.0, dtype=np.float32)
lh_sig_uncorr[lh_vidx] = rho_full[:n_lh] * sig_mask_uncorr[:n_lh]
rh_sig_uncorr = np.full(N_VERTS_RH, 0.0, dtype=np.float32)
rh_sig_uncorr[rh_vidx] = rho_full[n_lh:] * sig_mask_uncorr[n_lh:]

plot_surface_bilateral(
    lh_sig_uncorr, rh_sig_uncorr,
    title=f"{MODEL} / {MODALITY} — p<{ALPHA} uncorrected",
    threshold=1e-6, cmap="Reds",
)

In [ ]:
# ── Plot: FDR-corrected significance map ─────────────────────────────────────
sig_mask_fdr = pval_fdr < ALPHA  # (59412,) CIFTI space

lh_sig_fdr = np.full(N_VERTS_LH, 0.0, dtype=np.float32)
lh_sig_fdr[lh_vidx] = rho_full[:n_lh] * sig_mask_fdr[:n_lh]
rh_sig_fdr = np.full(N_VERTS_RH, 0.0, dtype=np.float32)
rh_sig_fdr[rh_vidx] = rho_full[n_lh:] * sig_mask_fdr[n_lh:]

plot_surface_bilateral(
    lh_sig_fdr, rh_sig_fdr,
    title=f"{MODEL} / {MODALITY} — FDR q<{ALPHA}",
    threshold=1e-6, cmap="Reds",
)

In [ ]:
# ── Build adjacency and find cluster borders ──────────────────────────────────
faces_lh = load_surface_faces(LEFT_SURF)
faces_rh = load_surface_faces(RIGHT_SURF)

# Adjacency in CIFTI grayordinate space (re-indexes 59k faces to drop medial wall)
adj_lh = cifti_surface_adjacency(faces_lh, lh_vidx, n_lh)
adj_rh = cifti_surface_adjacency(faces_rh, rh_vidx, n_rh)

# Significance masks in CIFTI grayordinate space
sig_lh = sig_mask_fdr[:n_lh]
sig_rh = sig_mask_fdr[n_lh:]

border_lh = find_cluster_borders(sig_lh, adj_lh, MIN_CLUSTER_SIZE)
border_rh = find_cluster_borders(sig_rh, adj_rh, MIN_CLUSTER_SIZE)

print(f"LH: {sig_lh.sum():,} significant, {border_lh.sum():,} border vertices")
print(f"RH: {sig_rh.sum():,} significant, {border_rh.sum():,} border vertices")

In [ ]:
# ── Plot: FDR map with cluster borders ───────────────────────────────────────
plot_surface_with_borders(
    lh_sig_fdr, rh_sig_fdr,
    border_lh, border_rh,
    lh_vidx, rh_vidx,
    title=f"{MODEL} / {MODALITY} — FDR q<{ALPHA}, clusters ≥{MIN_CLUSTER_SIZE} vertices",
    threshold=1e-6, cmap="Reds",
)

In [ ]:
# ── Pycortex flatmap (group average) ─────────────────────────────────────────
# pycortex subject hcp_999999_draw_NH uses 32k surfaces, so we resample the
# 59k CIFTI to 32k first using wb_command and the sphere registration surfaces.
import subprocess, tempfile

def pycortex_flatmap(cifti_path: str, title: str,
                     vmin: float = None, vmax: float = None,
                     cmap: str = "hot") -> None:
    """Resample 59k CIFTI to 32k and display as pycortex flatmap."""
    try:
        import cortex
    except ImportError:
        print("pycortex not installed — skipping flatmap")
        return

    # Resample 59k → 32k with wb_command barycentric surface resampling
    tmp = tempfile.mktemp(suffix=".dscalar.nii")
    result = subprocess.run([
        WORKBENCH, "-cifti-resample",
        cifti_path,   "COLUMN",
        TEMPLATE_32K, "COLUMN",
        "BARYCENTRIC", "ENCLOSING_VOXEL",
        tmp,
        "-left-spheres",  SPHERE_59K_LH, SPHERE_32K_LH,
        "-right-spheres", SPHERE_59K_RH, SPHERE_32K_RH,
    ], capture_output=True, text=True)

    if result.returncode != 0:
        print(f"wb_command resampling failed:\n{result.stderr}")
        return

    data_32k = nib.load(tmp).get_fdata(dtype=np.float32).squeeze()  # (59412,)

    cortex.database.default_filestore = str(HCP_DIR)
    lh_32k = data_32k[:29696]
    rh_32k = data_32k[29696:]
    # Expand to full 32k surface (medial wall = 0)
    lh_full_32k = np.zeros(32492, dtype=np.float32)
    rh_full_32k = np.zeros(32492, dtype=np.float32)
    bm32 = nib.load(TEMPLATE_32K).header.get_axis(1)
    for name, sl, struct in bm32.iter_structures():
        if "CORTEX_LEFT"  in name: lh_full_32k[struct.vertex] = data_32k[sl]
        elif "CORTEX_RIGHT" in name: rh_full_32k[struct.vertex] = data_32k[sl]

    vol = cortex.Volume(
        np.concatenate([lh_full_32k, rh_full_32k]),
        "hcp_999999_draw_NH", "MSMAll",
        vmin=vmin or 0,
        vmax=vmax or float(np.percentile(data_32k[data_32k > 0], 99)),
        cmap=cmap,
    )
    fig = cortex.quickflat.make_figure(vol, with_colorbar=True)
    fig.suptitle(title, fontsize=11)
    plt.show()

    import os; os.remove(tmp)


# Group average: rho map
pycortex_flatmap(str(GROUP_DIR / f"rsa_{METHOD}.dscalar.nii"),
                 f"{MODEL} / {MODALITY} — Spearman ρ (group average)",
                 vmin=0, cmap="hot")

# FDR-significant rho
pycortex_flatmap(str(GROUP_DIR / f"rsa_{METHOD}_fdr_sig.dscalar.nii"),
                 f"{MODEL} / {MODALITY} — FDR q<{ALPHA} (group average)",
                 vmin=1e-6, cmap="Reds")

## Per-Subject Random-Effects Analysis

For each vertex:
1. Load per-subject ρ maps
2. Fisher-z transform: z = arctanh(ρ)
3. One-sample t-test (H₀: mean z = 0); subjects = random effects
4. Cohen's d = mean(z) / std(z, ddof=1)
5. Whole-brain FDR correction on resulting p-values
6. Significance maps + borders
7. Cohen's d map

In [ ]:
# Collect per-subject ρ maps
subject_dirs = sorted([
    d for d in RSA_DIR.iterdir()
    if d.is_dir() and d.name not in ("group_average", "_geodesic_cache")
])

rho_maps = []
for sdir in subject_dirs:
    rho_file = sdir / MODEL / CONFIG / f"rsa_{METHOD}.dscalar.nii"
    if rho_file.exists():
        rho_maps.append(load_cifti_map(str(rho_file)))
    else:
        print(f"  MISSING: {rho_file}")

if not rho_maps:
    print("No per-subject maps found — run per-subject RSA first.")
else:
    rho_stack = np.stack(rho_maps, axis=0)  # (n_subjects, n_grayords)
    print(f"Loaded {len(rho_maps)} subjects  shape={rho_stack.shape}")

In [ ]:
# Fisher-z transform
z_stack = np.arctanh(np.clip(rho_stack, -1 + 1e-7, 1 - 1e-7)).astype(np.float32)

# One-sample t-test per vertex (H0: mean z = 0)
t_vals, p_vals = stats.ttest_1samp(z_stack, popmean=0.0, axis=0)
t_vals = t_vals.astype(np.float32)
p_vals = p_vals.astype(np.float32)

# One-tailed p (testing ρ > 0):
p_one_tailed = np.where(t_vals > 0, p_vals / 2.0, 1.0 - p_vals / 2.0).astype(np.float32)

# Cohen's d per vertex (using Fisher-z values)
mean_z = z_stack.mean(axis=0)
std_z  = z_stack.std(axis=0, ddof=1)
std_z[std_z == 0] = 1e-10
cohens_d = (mean_z / std_z).astype(np.float32)

print(f"t range: [{t_vals.min():.3f}, {t_vals.max():.3f}]")
print(f"Cohen's d range: [{cohens_d.min():.3f}, {cohens_d.max():.3f}]")
print(f"n(uncorr p<0.05): {(p_one_tailed < 0.05).sum():,}")

In [ ]:
# Whole-brain FDR correction on per-subject p-values
_, p_fdr_subj = fdrcorrection(p_one_tailed)
p_fdr_subj = p_fdr_subj.astype(np.float32)
print(f"FDR q<0.05: {(p_fdr_subj < ALPHA).sum():,} vertices")

# Save Cohen's d and FDR p-value maps
out_subj = RSA_DIR / "group_random_effects" / MODEL / CONFIG
out_subj.mkdir(parents=True, exist_ok=True)

save_cifti_map(cohens_d,   TEMPLATE_CIFTI, str(out_subj / "cohens_d.dscalar.nii"),  "cohens_d")
save_cifti_map(t_vals,     TEMPLATE_CIFTI, str(out_subj / "tstat.dscalar.nii"),     "tstat")
save_cifti_map(p_fdr_subj, TEMPLATE_CIFTI, str(out_subj / "pval_fdr.dscalar.nii"), "pval_fdr")

In [ ]:
# ── Per-subject FDR significance maps ────────────────────────────────────────
# Expand Cohen's d to full surface
lh_cohd, rh_cohd, _, _ = cifti_to_surface_arrays(
    cohens_d, TEMPLATE_CIFTI, N_VERTS_LH, N_VERTS_RH
)

# Significance in CIFTI grayordinate space
sig_lh_s = p_fdr_subj[:n_lh] < ALPHA   # (54216,)
sig_rh_s = p_fdr_subj[n_lh:] < ALPHA   # (54225,)

# Full-surface significance-masked Cohen's d (0 where not significant)
lh_stat_s = np.full(N_VERTS_LH, 0.0, dtype=np.float32)
lh_stat_s[lh_vidx] = cohens_d[:n_lh] * sig_lh_s
rh_stat_s = np.full(N_VERTS_RH, 0.0, dtype=np.float32)
rh_stat_s[rh_vidx] = cohens_d[n_lh:] * sig_rh_s

plot_surface_bilateral(
    lh_stat_s, rh_stat_s,
    title=f"{MODEL} / {MODALITY} — FDR q<{ALPHA}  (random effects, n={len(rho_maps)})",
    threshold=1e-6, cmap="Reds",
)

In [ ]:
# ── FDR map + cluster borders (per-subject) ───────────────────────────────────
border_lh_s = find_cluster_borders(sig_lh_s, adj_lh, MIN_CLUSTER_SIZE)
border_rh_s = find_cluster_borders(sig_rh_s, adj_rh, MIN_CLUSTER_SIZE)

print(f"LH: {sig_lh_s.sum():,} significant, {border_lh_s.sum():,} border vertices")
print(f"RH: {sig_rh_s.sum():,} significant, {border_rh_s.sum():,} border vertices")

plot_surface_with_borders(
    lh_stat_s, rh_stat_s,
    border_lh_s, border_rh_s,
    lh_vidx, rh_vidx,
    title=f"{MODEL} / {MODALITY} — FDR q<{ALPHA}, clusters ≥{MIN_CLUSTER_SIZE}  (n={len(rho_maps)} subjects)",
    threshold=1e-6, cmap="Reds",
)

In [ ]:
# ── Cohen's d map (all vertices) ─────────────────────────────────────────────
d_pos = cohens_d[cohens_d > 0]
vmax_d = float(np.percentile(d_pos, 99)) if len(d_pos) else 1.0

plot_surface_bilateral(
    lh_cohd, rh_cohd,
    title=f"{MODEL} / {MODALITY} — Cohen's d  (n={len(rho_maps)} subjects)",
    threshold=0.0, cmap="hot", vmax=vmax_d,
)

In [ ]:
# ── Cohen's d — only FDR-significant vertices, with cluster borders ───────────
lh_cohd_sig = np.full(N_VERTS_LH, 0.0, dtype=np.float32)
lh_cohd_sig[lh_vidx] = cohens_d[:n_lh] * sig_lh_s
rh_cohd_sig = np.full(N_VERTS_RH, 0.0, dtype=np.float32)
rh_cohd_sig[rh_vidx] = cohens_d[n_lh:] * sig_rh_s

plot_surface_with_borders(
    lh_cohd_sig, rh_cohd_sig,
    border_lh_s, border_rh_s,
    lh_vidx, rh_vidx,
    title=f"{MODEL} / {MODALITY} — Cohen's d (FDR q<{ALPHA}, clusters ≥{MIN_CLUSTER_SIZE})",
    threshold=1e-6, cmap="hot",
)

In [ ]:
# ── Pycortex flatmaps (random-effects) ───────────────────────────────────────
out_subj = RSA_DIR / "group_random_effects" / MODEL / CONFIG

pycortex_flatmap(str(out_subj / "cohens_d.dscalar.nii"),
                 f"{MODEL} / {MODALITY} — Cohen's d  (n={len(rho_maps)} subjects)",
                 vmin=0, cmap="hot")

pycortex_flatmap(str(out_subj / "pval_fdr.dscalar.nii"),
                 f"{MODEL} / {MODALITY} — FDR p-value  (n={len(rho_maps)} subjects)",
                 vmin=0, vmax=ALPHA, cmap="Reds_r")